# Kitchenware YOLOv8 — upload zip (no Google Drive mount)

1. On your PC use: `data/processed/object_detect/kitchenware_yolo_dataset.zip`
2. Runtime → **T4 GPU**
3. Run all cells — when asked, **upload the zip** (~570 MB)
4. Download `best.pt` and **`training_history.csv`** (full epoch metrics)
5. Save weights to `ml/models/object_detect/kitchenware/best.pt` and CSV beside it as `training_history.csv`

In [ ]:
!pip install -q ultralytics

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

print("Select kitchenware_yolo_dataset.zip from your PC...")
uploaded = files.upload()
zip_path = Path(next(iter(uploaded)))

extract = Path("/content/kitchenware_dataset")
if extract.exists():
    import shutil
    shutil.rmtree(extract)
extract.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract)

if (extract / "data.yaml").is_file():
    DATA_DIR = extract
else:
    DATA_DIR = next(p.parent for p in extract.rglob("data.yaml"))

yaml = DATA_DIR / "data.yaml"
yaml.write_text(
    f"path: {DATA_DIR.as_posix()}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    "nc: 8\n"
    "names: [plate, pot, soap, water_dipper, mug, bowl, knife, spoon]\n",
    encoding="utf-8",
)

TRAIN_DATA = str(yaml)
print("DATA_DIR:", DATA_DIR)
print("Train images:", len(list((DATA_DIR / "train/images").glob("*.jpg"))))
print(yaml.read_text())

In [ ]:
from ultralytics import YOLO

RUNS = Path("/content/runs/yolo")
RUNS.mkdir(parents=True, exist_ok=True)
NAME = "kitchenware_yolov8m_colab"

YOLO("yolov8m.pt").train(
    data=TRAIN_DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    close_mosaic=10,
    project=str(RUNS),
    name=NAME,
    exist_ok=True,
    plots=True,
    val=True,
)

RUN_DIR = RUNS / NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
HISTORY_CSV = RUN_DIR / "results.csv"
print("best.pt:", BEST_PT)
print("training history:", HISTORY_CSV)

In [ ]:
import pandas as pd
from IPython.display import Image, display

history = pd.read_csv(HISTORY_CSV)
display(history.tail(10))
print(f"epochs logged: {len(history)}")

archive = Path("/content/kitchenware_training_history.csv")
history.to_csv(archive, index=False)

rp = RUN_DIR / "results.png"
if rp.is_file():
    display(Image(filename=str(rp), width=900))
print(YOLO(str(BEST_PT)).val(data=TRAIN_DATA, imgsz=640, device=0))

In [ ]:
files.download(str(BEST_PT))
files.download(str(HISTORY_CSV))
files.download(str(archive))